In [14]:
import pandas as pd

# Load the dataset
df = pd.read_csv("insurance.csv")

# Display the first few rows
df.head()


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [15]:
import pandas as pd

# Load the dataset
df = pd.read_csv("insurance.csv")

# Display the first few rows
df.head()


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [16]:
# Convert each row into a JSON-like string for context-rich embeddings
documents = df.apply(lambda row: row.to_json(), axis=1).tolist()

# View sample
print(documents[0])


{"age":19,"sex":"female","bmi":27.9,"children":0,"smoker":"yes","region":"southwest","charges":16884.924}


In [17]:
from sentence_transformers import SentenceTransformer

# Load a lightweight transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
embeddings = model.encode(documents, show_progress_bar=True)


Batches: 100%|█████████████████████████████████████████████████████████████████████████| 42/42 [00:20<00:00,  2.08it/s]


In [18]:
import faiss
import numpy as np
import pickle

# Define vector dimension
dimension = embeddings.shape[1]

# Create FAISS index
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# Save index and metadata
faiss.write_index(index, "insurance_faiss.index")
with open("insurance_metadata.pkl", "wb") as f:
    pickle.dump(documents, f)

print("✅ FAISS index & metadata stored successfully.")


✅ FAISS index & metadata stored successfully.


In [19]:
def search_faiss(query, top_k=5):
    # Load the model, index, and metadata
    index = faiss.read_index("insurance_faiss.index")
    with open("insurance_metadata.pkl", "rb") as f:
        docs = pickle.load(f)
    
    # Embed the query
    query_vector = model.encode([query])
    
    # Search FAISS index
    distances, indices = index.search(np.array(query_vector), top_k)
    
    # Return results
    return [docs[i] for i in indices[0]]


In [20]:
query = "person who is a smoker and pays high insurance"
results = search_faiss(query, top_k=3)

for i, res in enumerate(results):
    print(f"\nResult {i+1}:\n{res}")



Result 1:
{"age":62,"sex":"female","bmi":26.29,"children":0,"smoker":"yes","region":"southeast","charges":27808.7251}

Result 2:
{"age":34,"sex":"male","bmi":22.42,"children":2,"smoker":"no","region":"northeast","charges":27375.90478}

Result 3:
{"age":62,"sex":"male","bmi":27.55,"children":1,"smoker":"no","region":"northwest","charges":13937.6665}
